# Day 4 — Series and DataFrame: schema, no index

*100 Days of Polars · Phase 1: Foundations and mental model*

- **Post:** [Series and DataFrame: Strict Types, No Index](https://medium.com/python-in-plain-english/day-4-series-and-dataframe-strict-types-no-index-8a4030db52ae)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [ ]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


Two building blocks: a **Series** is one typed column, a **DataFrame** is a set of equal-length Series with names.
What is *missing* matters as much: there is no index.

## 1. Series: one name, one dtype

In [ ]:
prices = pl.Series("price", [58.90, 239.90, 199.00])
print(prices)
print(prices.dtype, prices.len())

shape: (3,)
Series: 'price' [f64]
[
	58.9
	239.9
	199.0
]
Float64 3


In [ ]:
# Types are strict: mixed types do not silently become 'object'
try:
    pl.Series("mixed", [1, "two", 3.0])
except Exception as err:
    print(type(err).__name__, "->", err)

print(pl.Series("mixed", [1, "two", 3.0], strict=False))  # opt in to a common type

TypeError -> unexpected value while building Series of type Int64; found value of type String: "two"

This error occurred with the following context stack:
	[1] while constructing Series 'mixed'


Hint: Try setting `strict=False` to allow passing data with mixed types.
shape: (3,)
Series: 'mixed' [str]
[
	"1"
	"two"
	"3.0"
]


In [ ]:
# Missing values are null, for every dtype
pl.Series("installments", [1, None, 3], dtype=pl.Int16)

installments
i16
1
null
3


## 2. DataFrame: named Series of equal length

In [ ]:
df = pl.DataFrame(
    {
        "order_id": ["a1", "b2", "c3"],
        "status": ["delivered", "shipped", "delivered"],
        "price": [58.90, 239.90, 199.00],
    }
)
df

order_id,status,price
str,str,f64
"""a1""","""delivered""",58.9
"""b2""","""shipped""",239.9
"""c3""","""delivered""",199.0


In [ ]:
print(df.schema)   # name -> dtype, the contract of the frame
print(df.columns)
print(df.dtypes)
print(df.shape)

Schema({'order_id': String, 'status': String, 'price': Float64})
['order_id', 'status', 'price']
[String, String, Float64]
(3, 3)


In [ ]:
# Declare the schema up front when you know it
pl.DataFrame(
    {"order_id": ["a1"], "installments": [3]},
    schema={"order_id": pl.String, "installments": pl.UInt8},
)

order_id,installments
str,u8
"""a1""",3


## 3. No index — and why that is a feature

In pandas, arithmetic between two Series **aligns on the index**. That is powerful, and also a classic source of silent NaNs.

In [ ]:
import pandas as pd

a = pd.Series([10, 20, 30], index=[0, 1, 2])
b = pd.Series([1, 2, 3], index=[1, 2, 3])
print(a + b)  # aligned on index labels -> NaN at 0 and 3

0     NaN
1    21.0
2    32.0
3     NaN
dtype: float64


In [ ]:
# Polars works by position: same length in, same length out
pl.Series([10, 20, 30]) + pl.Series([1, 2, 3])

""
i64
11
22
33


Need a row number? Make it an explicit column. Need a lookup by key? Use `filter` or a `join`.

In [ ]:
orders = pl.read_csv("data/raw/olist_orders_dataset.csv", try_parse_dates=True)

orders.with_row_index("row_nr").head(3)

row_nr,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
u32,str,str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs]
0,"""e481f51cbdc54678b7cc49136f2d6a…","""9ef432eb6251297304e76186b10a92…","""delivered""",2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,"""53cdb2fc8bc7dce0b6741e21502734…","""b0830fb4747a6c6d20dea0b8c802d7…","""delivered""",2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,"""47770eb9100c2d0c44946d9cf07ec6…","""41ce2a54c0b03bf3443c3d931a3670…","""delivered""",2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [ ]:
# "loc by key" becomes a filter
orders.filter(pl.col("order_id") == "e481f51cbdc54678b7cc49136f2d6af7")

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs]
"""e481f51cbdc54678b7cc49136f2d6a…","""9ef432eb6251297304e76186b10a92…","""delivered""",2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00


## 4. First look at a real table

In [ ]:
orders.glimpse()

Rows: 99441
Columns: 8
$ order_id                               <str> 'e481f51cbdc54678b7cc49136f2d6af7', '53cdb2fc8bc7dce0b6741e2150273451', '47770eb9100c2d0c44946d9cf07ec65d', '949d5b44dbf5de918fe9c16f97b45f8a', 'ad21c59c0840e6cb83a9ceb5573f8159', 'a4591c265e18cb1dcee52889e2d8acc3', '136cce7faa42fdb2cefd53fdc79a6098', '6514b8ad8028c9f2cc2374ded245783f', '76c6e866289321a7c93b82b54852dc33', 'e69bfb5eb88e0ed6a785585b27e16dbf'
$ customer_id                            <str> '9ef432eb6251297304e76186b10a928d', 'b0830fb4747a6c6d20dea0b8c802d7ef', '41ce2a54c0b03bf3443c3d931a367089', 'f88197465ea7920adcdbec7375364d82', '8ab97904e6daea8866dbdbc4fb7aad2c', '503740e9ca751ccdda7ba28e9ab8f608', 'ed0271e0b7da060a393796590e7b737a', '9bdf08b4b3b52b5526ff42d37d47f222', 'f54a9f0e6b351c431402b8461ea51999', '31ad1d1b63eb9962463f764d4e6e0c9d'
$ order_status                           <str> 'delivered', 'delivered', 'delivered', 'delivered', 'delivered', 'delivered', 'invoiced', 'delivered', 'delivered', 'd

In [ ]:
orders.describe()

statistic,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,str,str,str,str,str,str
"""count""","""99441""","""99441""","""99441""","""99441""","""99281""","""97658""","""96476""","""99441"""
"""null_count""","""0""","""0""","""0""","""0""","""160""","""1783""","""2965""","""0"""
"""mean""",null,null,null,"""2017-12-31 08:43:12.776581""","""2017-12-31 18:35:24.098800""","""2018-01-04 21:49:48.138278""","""2018-01-14 12:09:19.035542""","""2018-01-24 03:08:37.730111"""
"""std""",null,null,null,null,null,null,null,null
"""min""","""00010242fe8c5a6d1ba2dd792cb162…","""00012a2ce6f8dcda20d059ce984917…","""approved""","""2016-09-04 21:15:19""","""2016-09-15 12:16:38""","""2016-10-08 10:34:01""","""2016-10-11 13:46:32""","""2016-09-30 00:00:00"""
"""25%""",null,null,null,"""2017-09-12 14:46:19""","""2017-09-12 23:24:16""","""2017-09-15 22:27:50""","""2017-09-25 22:07:34""","""2017-10-03 00:00:00"""
"""50%""",null,null,null,"""2018-01-18 23:04:36""","""2018-01-19 11:36:13""","""2018-01-24 16:12:38""","""2018-02-02 19:28:30""","""2018-02-15 00:00:00"""
"""75%""",null,null,null,"""2018-05-04 15:42:16""","""2018-05-04 20:35:10""","""2018-05-08 13:38:00""","""2018-05-15 22:48:50""","""2018-05-25 00:00:00"""
"""max""","""fffe41c64501cc87c801fd61db3f62…","""ffffe8b65bbe3087b653a978c870db…","""unavailable""","""2018-10-17 17:30:18""","""2018-09-03 17:40:06""","""2018-09-11 19:48:28""","""2018-10-17 13:22:46""","""2018-11-12 00:00:00"""


In [ ]:
# Getting at values
print(orders["order_status"].head(3))   # a Series
print(orders.row(0))                     # one row as a tuple
print(orders.select(pl.len()).item())    # a single scalar

shape: (3,)
Series: 'order_status' [str]
[
	"delivered"
	"delivered"
	"delivered"
]
('e481f51cbdc54678b7cc49136f2d6af7', '9ef432eb6251297304e76186b10a928d', 'delivered', datetime.datetime(2017, 10, 2, 10, 56, 33), datetime.datetime(2017, 10, 2, 11, 7, 15), datetime.datetime(2017, 10, 4, 19, 55), datetime.datetime(2017, 10, 10, 21, 25, 13), datetime.datetime(2017, 10, 18, 0, 0))
99441


## 5. Moving between pandas and Polars

In [ ]:
orders_pd = orders.to_pandas()
print(type(orders_pd), orders_pd.shape)

back = pl.from_pandas(orders_pd)
print(back.shape == orders.shape)
back.schema  # compare with orders.schema: check the datetime precision after the round trip

<class 'pandas.DataFrame'> (99441, 8)
True


Schema([('order_id', String),
        ('customer_id', String),
        ('order_status', String),
        ('order_purchase_timestamp', Datetime(time_unit='us', time_zone=None)),
        ('order_approved_at', Datetime(time_unit='us', time_zone=None)),
        ('order_delivered_carrier_date',
         Datetime(time_unit='us', time_zone=None)),
        ('order_delivered_customer_date',
         Datetime(time_unit='us', time_zone=None)),
        ('order_estimated_delivery_date',
         Datetime(time_unit='us', time_zone=None))])

## Try it yourself

1. Build a DataFrame with a column of `None` only. What dtype does Polars pick? How do you set it?
2. In pandas, `orders_pd.set_index("order_id")` and look up one order with `.loc`. Write the Polars equivalent without an index.
3. What does `orders.schema` say about `order_purchase_timestamp` with and without `try_parse_dates=True`?